# Fake News Detection — Automatic Model Selection

This notebook rebuilds the ML part of the project from the existing training data.

## What changed

- Keeps the existing **TF-IDF + classical ML** approach.
- Uses **RandomizedSearchCV**, not Optuna.
- Tunes Logistic Regression, Linear SVM, Random Forest, and Multinomial Naive Bayes automatically.
- Selects the best model using **cross-validated Macro-F1** on a tuning subset.
- Evaluates the selected model once on an untouched test set.
- Calibrates the selected classifier so the Streamlit app can always obtain `predict_proba`.
- Saves the vectorizer, selected model, metadata, and tuning results.

The test set is never used for hyperparameter selection.

In [7]:
# ============================================================
# 1. IMPORTS
# ============================================================

import json
import os
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATA_DIR = "data"
MODEL_DIR = "models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Imports completed.")

Imports completed.


In [1]:
pip install -U ipykernel

  Attempting uninstall: ipykernel
    Found existing installation: ipykernel 7.3.0
    Uninstalling ipykernel-7.3.0:
      Successfully uninstalled ipykernel-7.3.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# ============================================================
# 2. DATA LOADING
# ============================================================

# data_fake = pd.read_csv(
#     os.path.join(DATA_DIR, "Fake.csv")
# )

# data_true = pd.read_csv(
#     os.path.join(DATA_DIR, "True.csv")
# )

data_news1 = pd.read_csv(
    os.path.join(DATA_DIR, "news.csv")
)

data_news2 = pd.read_csv(
    os.path.join(DATA_DIR, "news2.csv")
)

# print("Fake.csv :", data_fake.shape)
# print("True.csv :", data_true.shape)
print("news.csv :", data_news1.shape)
print("news2.csv:", data_news2.shape)

news.csv : (44898, 2)
news2.csv: (45757, 2)


In [9]:
# ============================================================
# 3. STANDARDIZE EACH DATASET
# ============================================================

def standardize_dataset(df, fixed_label=None):
    df = df.copy()

    # Find a text column.
    if "text" not in df.columns:
        if "title" in df.columns:
            df["text"] = df["title"].fillna("")
        else:
            raise ValueError(
                "Dataset does not contain a usable text column."
            )

    # Create/normalize the target label.
    if fixed_label is not None:
        df["label"] = fixed_label
    elif "label" in df.columns:
        label_series = df["label"]

        # Numeric labels.
        numeric = pd.to_numeric(
            label_series,
            errors="coerce",
        )

        if numeric.notna().all():
            df["label"] = numeric.astype(int)
        else:
            mapping = {
                "fake": 0,
                "false": 0,
                "real": 1,
                "true": 1,
            }

            df["label"] = (
                label_series
                .astype(str)
                .str.strip()
                .str.lower()
                .map(mapping)
            )

    if "label" not in df.columns:
        raise ValueError(
            "Dataset does not contain a usable label column."
        )

    # Use title + body when title exists.
    # This helps the deployed model handle headlines.
    if "title" in df.columns:
        title = df["title"].fillna("").astype(str)
        body = df["text"].fillna("").astype(str)

        df["text"] = (
            title + " " + body
        ).str.strip()

    df = df[["text", "label"]].copy()

    df["text"] = (
        df["text"]
        .fillna("")
        .astype(str)
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce",
    )

    df = df.dropna(
        subset=["text", "label"]
    )

    df["label"] = df["label"].astype(int)

    # Keep only binary labels used by this project.
    df = df[
        df["label"].isin([0, 1])
    ]

    return df




news1_df = standardize_dataset(
    data_news1,
)

news2_df = standardize_dataset(
    data_news2,
)


print("News 1:", news1_df.shape)
print("News 2:", news2_df.shape)

News 1: (44898, 2)
News 2: (45757, 2)


In [10]:
# ============================================================
# 4. COMBINE + CLEAN + DEDUPLICATE
# ============================================================

final_data = pd.concat(
    [
        news1_df,
        news2_df,
    ],
    ignore_index=True,
)

# Remove empty text.
final_data["text"] = (
    final_data["text"]
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

final_data = final_data[
    final_data["text"].str.len() >= 20
].copy()

# Remove exact duplicate articles.
final_data = final_data.drop_duplicates(
    subset=["text"],
    keep="first",
).reset_index(drop=True)

print("Final dataset shape:", final_data.shape)
print("\nClass distribution:")
print(final_data["label"].value_counts())

Final dataset shape: (84360, 2)

Class distribution:
label
1    44084
0    40276
Name: count, dtype: int64


In [11]:
# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

X = final_data["text"]
y = final_data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 67488
Test samples: 16872


## Why the vectorizer is fitted only on training data

The TF-IDF vocabulary and IDF weights must not see the test set. Otherwise the test set leaks information into training.

The vectorizer below is fitted only on `X_train`, then reused unchanged for validation/tuning and the final test.

In [12]:
# ============================================================
# 6. TF-IDF FEATURE ENGINEERING
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=80000,
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
)

X_train_tfidf = vectorizer.fit_transform(
    X_train
)

X_test_tfidf = vectorizer.transform(
    X_test
)

print(
    "TF-IDF training matrix:",
    X_train_tfidf.shape,
)

print(
    "TF-IDF test matrix:",
    X_test_tfidf.shape,
)


TF-IDF training matrix: (67488, 80000)
TF-IDF test matrix: (16872, 80000)


## Automatic hyperparameter tuning

We use `RandomizedSearchCV` rather than manually choosing a winner.

The search compares:

1. Logistic Regression
2. Linear SVM
3. Random Forest
4. Multinomial Naive Bayes

The selection metric is **Macro-F1**, which gives equal importance to the Fake and Real classes.

To keep the search practical on the 100K+ article dataset, hyperparameter search uses a stratified tuning subset of the training data. After the winner is selected, it is refit on the full training set.

In [14]:
TUNE_MAX_SAMPLES = 40000

# For sparse matrices, use shape[0] instead of len()
n_train_samples = X_train_tfidf.shape[0]

if n_train_samples > TUNE_MAX_SAMPLES:

    X_tune, _, y_tune, _ = train_test_split(
        X_train_tfidf,
        y_train,
        train_size=TUNE_MAX_SAMPLES,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )

else:

    X_tune = X_train_tfidf
    y_tune = y_train

print("Tuning samples:", X_tune.shape[0])
print("Tuning matrix shape:", X_tune.shape)

Tuning samples: 40000
Tuning matrix shape: (40000, 80000)


In [15]:
# ============================================================
# 8. MODEL SEARCH SPACES
# ============================================================

search_spaces = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
        ),
        {
            "C": [
                0.05,
                0.1,
                0.3,
                1.0,
                3.0,
                10.0,
            ],
            "class_weight": [
                None,
                "balanced",
            ],
            "solver": [
                "liblinear",
                "lbfgs",
            ],
        },
    ),

    "Linear SVM": (
        LinearSVC(
            random_state=RANDOM_STATE,
        ),
        {
            "C": [
                0.05,
                0.1,
                0.3,
                1.0,
                3.0,
                10.0,
            ],
            "class_weight": [
                None,
                "balanced",
            ],
        },
    ),

    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        {
            "n_estimators": [
                100,
                200,
                300,
            ],
            "max_depth": [
                None,
                20,
                40,
            ],
            "min_samples_split": [
                2,
                5,
                10,
            ],
            "min_samples_leaf": [
                1,
                2,
            ],
            "max_features": [
                "sqrt",
                0.2,
            ],
            "class_weight": [
                None,
                "balanced",
            ],
        },
    ),

    "Multinomial Naive Bayes": (
        MultinomialNB(),
        {
            "alpha": [
                0.01,
                0.05,
                0.1,
                0.5,
                1.0,
                2.0,
            ],
            "fit_prior": [
                True,
                False,
            ],
        },
    ),
}

print(
    "Models to compare:",
    list(search_spaces.keys()),
)

Models to compare: ['Logistic Regression', 'Linear SVM', 'Random Forest', 'Multinomial Naive Bayes']


In [16]:
# ============================================================
# 9. RUN AUTOMATIC SEARCH FOR EVERY MODEL
# ============================================================

search_results = []
best_estimators = {}

for model_name, (
    estimator,
    param_distributions,
) in search_spaces.items():

    print("\n" + "=" * 70)
    print("TUNING:", model_name)
    print("=" * 70)

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=8,
        scoring="f1_macro",
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )

    search.fit(
        X_tune,
        y_tune,
    )

    best_estimators[model_name] = (
        search.best_estimator_
    )

    search_results.append(
        {
            "model": model_name,
            "cv_macro_f1": float(
                search.best_score_
            ),
            "best_params": str(
                search.best_params_
            ),
        }
    )

    print(
        "Best CV Macro-F1:",
        round(
            search.best_score_,
            5,
        ),
    )

    print(
        "Best parameters:",
        search.best_params_,
    )

results_df = pd.DataFrame(
    search_results
).sort_values(
    "cv_macro_f1",
    ascending=False,
).reset_index(drop=True)

results_df


TUNING: Logistic Regression
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV Macro-F1: 0.98675
Best parameters: {'solver': 'liblinear', 'class_weight': 'balanced', 'C': 3.0}

TUNING: Linear SVM
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV Macro-F1: 0.99201
Best parameters: {'class_weight': 'balanced', 'C': 10.0}

TUNING: Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV Macro-F1: 0.98971
Best parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.2, 'max_depth': 40, 'class_weight': 'balanced'}

TUNING: Multinomial Naive Bayes
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV Macro-F1: 0.95986
Best parameters: {'fit_prior': False, 'alpha': 0.01}


,model,cv_macro_f1,best_params
0,Linear SVM,0.992009,"{'class_weight': 'balanced', 'C': 10.0}"
1,Random Forest,0.989708,"{'n_estimators': 100, 'min_samples_split': 5, ..."
2,Logistic Regression,0.986748,"{'solver': 'liblinear', 'class_weight': 'balan..."
3,Multinomial Naive Bayes,0.959863,"{'fit_prior': False, 'alpha': 0.01}"


## Automatic model selection

The following cell selects the model with the highest cross-validated Macro-F1.

No model is manually selected.

In [17]:
# ============================================================
# 10. SELECT THE BEST MODEL AUTOMATICALLY
# ============================================================

best_model_name = results_df.iloc[0]["model"]

best_model = best_estimators[
    best_model_name
]

best_cv_f1 = float(
    results_df.iloc[0]["cv_macro_f1"]
)

print("Automatically selected model:")
print(best_model_name)

print(
    "Best CV Macro-F1:",
    round(
        best_cv_f1,
        5,
    ),
)

Automatically selected model:
Linear SVM
Best CV Macro-F1: 0.99201


## Refit the selected model on all training data

The hyperparameter search was performed on a tuning subset. Now the selected configuration is refit on the complete training split.

In [18]:
# ============================================================
# 11. REFIT WINNER ON FULL TRAINING DATA
# ============================================================

best_model.fit(
    X_train_tfidf,
    y_train,
)

print(
    "Selected model refitted on full training data."
)

Selected model refitted on full training data.


## Probability calibration

Linear SVM does not natively provide probabilities. Because the Streamlit application should always show Fake/Real probabilities, the selected classifier is wrapped with `CalibratedClassifierCV`.

The calibration uses cross-validation and produces `predict_proba`.

In [19]:
# ============================================================
# 12. CALIBRATE THE SELECTED MODEL
# ============================================================

try:
    calibrated_model = CalibratedClassifierCV(
        estimator=best_model,
        method="sigmoid",
        cv=3,
        n_jobs=-1,
    )
except TypeError:
    # Compatibility with older scikit-learn versions.
    calibrated_model = CalibratedClassifierCV(
        base_estimator=best_model,
        method="sigmoid",
        cv=3,
    )

calibrated_model.fit(
    X_train_tfidf,
    y_train,
)

print(
    "Calibrated model ready."
)

Calibrated model ready.


In [20]:
# ============================================================
# 13. FINAL TEST EVALUATION
# ============================================================

test_predictions = calibrated_model.predict(
    X_test_tfidf
)

test_probabilities = (
    calibrated_model.predict_proba(
        X_test_tfidf
    )
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

test_precision = precision_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0,
)

test_recall = recall_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0,
)

test_f1_macro = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0,
)

print(
    "FINAL TEST RESULTS"
)

print(
    "Accuracy :",
    round(test_accuracy, 5),
)

print(
    "Precision:",
    round(test_precision, 5),
)

print(
    "Recall   :",
    round(test_recall, 5),
)

print(
    "Macro-F1 :",
    round(test_f1_macro, 5),
)

print(
    "\nClassification report:\n"
)

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "Fake",
            "Real",
        ],
        zero_division=0,
    )
)

FINAL TEST RESULTS
Accuracy : 0.99484
Precision: 0.99472
Recall   : 0.99496
Macro-F1 : 0.99483

Classification report:

              precision    recall  f1-score   support

        Fake       0.99      1.00      0.99      8055
        Real       1.00      0.99      1.00      8817

    accuracy                           0.99     16872
   macro avg       0.99      0.99      0.99     16872
weighted avg       0.99      0.99      0.99     16872



In [21]:
# ============================================================
# 14. SAVE MODEL ARTIFACTS
# ============================================================

joblib.dump(
    vectorizer,
    os.path.join(
        MODEL_DIR,
        "tfidf_vectorizer.pkl",
    ),
)

joblib.dump(
    calibrated_model,
    os.path.join(
        MODEL_DIR,
        "best_model.pkl",
    ),
)

results_df.to_csv(
    os.path.join(
        MODEL_DIR,
        "model_search_results.csv",
    ),
    index=False,
)

metadata = {
    "best_model_name": best_model_name,
    "best_cv_macro_f1": best_cv_f1,
    "test_accuracy": float(test_accuracy),
    "test_precision_macro": float(test_precision),
    "test_recall_macro": float(test_recall),
    "test_f1_macro": float(test_f1_macro),
    "vectorizer": {
        "type": "TfidfVectorizer",
        "stop_words": "english",
        "ngram_range": [1, 2],
        "max_features": 80000,
        "min_df": 2,
        "max_df": 0.98,
        "sublinear_tf": True,
    },
    "class_mapping": {
        "0": "Fake",
        "1": "Real",
    },
    "random_state": RANDOM_STATE,
}

with open(
    os.path.join(
        MODEL_DIR,
        "ml_metadata.json",
    ),
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=4,
    )

print(
    "\nAll model artifacts saved to:",
    MODEL_DIR,
)


All model artifacts saved to: models


In [22]:
# ============================================================
# 15. TEST ONE SAMPLE
# ============================================================

def predict_news(text):
    cleaned = str(text).lower()

    cleaned = re.sub(
        r"https?://\S+",
        " ",
        cleaned,
    )

    cleaned = re.sub(
        r"www\.\S+",
        " ",
        cleaned,
    )

    cleaned = re.sub(
        r"<.*?>",
        " ",
        cleaned,
    )

    cleaned = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        cleaned,
    )

    cleaned = re.sub(
        r"\s+",
        " ",
        cleaned,
    ).strip()

    features = vectorizer.transform(
        [cleaned]
    )

    prediction = int(
        calibrated_model.predict(
            features
        )[0]
    )

    probabilities = (
        calibrated_model.predict_proba(
            features
        )[0]
    )

    return {
        "prediction": (
            "REAL"
            if prediction == 1
            else "FAKE"
        ),
        "fake_probability": float(
            probabilities[0]
        ),
        "real_probability": float(
            probabilities[1]
        ),
    }


sample_text = (
    "The government announced a new policy "
    "to improve public transportation."
)

predict_news(sample_text)

{'prediction': 'FAKE',
 'fake_probability': 0.9672301404100274,
 'real_probability': 0.03276985958997266}

## Files produced

After the notebook finishes, the project should contain:

```text
models/
├── tfidf_vectorizer.pkl
├── best_model.pkl
├── ml_metadata.json
└── model_search_results.csv
```

`app.py` loads `tfidf_vectorizer.pkl` and `best_model.pkl`. It does **not** manually choose Logistic Regression anymore.